# E04:
we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

In [2]:
# import
import torch

In [3]:
# split data set
words = open('../../../data/name.txt', 'r').read().splitlines()
train_words = words

# build stoi and itos
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

In [4]:
# bigram model
def parse_bigram_data(words):
    # create the training set of bigrams
    xs, ys = [], []
    for w in words:
        chs = ["."] + list(w) + ["."]
        for ch1, ch2 in zip(chs, chs[1:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            xs.append(ix1)
            ys.append(ix2)

    xs = torch.tensor(xs)
    ys = torch.tensor(ys)

    return xs, ys

def train_bigram_model(n_iter=100):

    xs, ys = parse_bigram_data(train_words)
    n_train_data = len(xs)

    # initialize the network
    g = torch.Generator().manual_seed(2147483647)
    W = torch.randn((27, 27), generator=g, requires_grad=True)

    # gradient descent
    for k in range(n_iter):
        # forward pass
        logits = W[xs]
        counts = logits.exp()  # equivalent to N
        probs = counts / counts.sum(1, keepdim=True)  # probabilities for next character
        # L2 regularization
        # loss = -probs[torch.arange(n_train_data), ys].log().mean() + 0.1 * (W**2).mean()
        loss = -probs[torch.arange(n_train_data), ys].log().mean()
        if k == n_iter - 1:
            print("train loss: ", loss.item())

        # backward pass
        W.grad = None  # set to zero the gardient
        loss.backward()

        # update
        W.data += -50 * W.grad

    return W


@torch.no_grad()
def evaluate_bigram_model(W, evaluate_words):
    xs, ys = parse_bigram_data(evaluate_words)
    n = len(xs)

    logits = W[xs]  # the 1-hot @ W is just a row select, so index W directly
    counts = logits.exp()  # equivalent to N
    probs = counts / counts.sum(1, keepdim=True)  # probabilities for next character
    loss = -probs[torch.arange(n), ys].log().mean()

    return loss.item()

In [5]:
W_bi = train_bigram_model(n_iter=100)

train loss:  2.4728763103485107
